In [5]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
import os
os.chdir(r"C:\Kaggle_Competition\Playground\S6E6-STELLAR-CLASS")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import QMCSampler, TPESampler
import config
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder
optuna.logging.set_verbosity(verbosity=optuna.logging.WARNING)

import warnings
from optuna.exceptions import ExperimentalWarning
warnings.filterwarnings("ignore", category=ExperimentalWarning)

In [7]:
train = pd.read_csv("data/raw/train.csv")
test = pd.read_csv("data/raw/test.csv")

train_ids = train[config.ID_COL].values
test_ids = test[config.ID_COL].values

y = train[config.TARGET_COL]

enc = LabelEncoder()
y = enc.fit_transform(y)

print(f"Shape of train data: {train.shape}")
print(f"Shape of test data: {test.shape}")

Shape of train data: (577347, 12)
Shape of test data: (247435, 11)


### Threshold Tuning

In [ ]:
MODELS = [
    ("histgbm", r"artifacts\oof_proba\histgbm_v1_seed42_oof_proba.csv", r"artifacts\test_proba\histgbm_v1_seed42_test_proba.csv"), 
    ("lgbm", r"artifacts\oof_proba\lightgbm_v3_seed42_oof_proba.csv", r"artifacts\test_proba\lightgbm_v3_seed42_test_proba.csv"),
    ("xgbm", r"artifacts\oof_proba\xgbm_v3_seed42_oof_proba.csv", r"artifacts\test_proba\xgbm_v3_seed42_test_proba.csv"),
    ("realmlp", r'artifacts\oof_proba\realmlp_v0_seed42_oof_proba.csv', r'artifacts\test_proba\realmlp_v0_seed42_test_proba.csv'),
]

def load_pred(path: str, prob_cols=config.PROB_COLS) -> np.ndarray:
    df = pd.read_csv(path)

    if all(col in df.columns for col in prob_cols):
        return df[prob_cols].to_numpy(dtype=np.float32)

    num_df = df.select_dtypes(include=[np.number])
    if num_df.shape[1] >= 3:
        return num_df.iloc[:, -3:].to_numpy(dtype=np.float32)

    raise ValueError(f"Could not find 3 probability columns in: {path}")


def softmax(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    e = np.exp(x - x.max())
    return e / e.sum()


def weighted_blend(weights: np.ndarray, matrices: list[np.ndarray]) -> np.ndarray:
    """
    matrices: list of shape (n_samples, 3)
    weights: shape (n_models,)
    returns: shape (n_samples, 3)
    """
    blend = np.zeros_like(matrices[0], dtype=np.float32)
    for w, mat in zip(weights, matrices):
        blend += w * mat
    return blend


names = [m[0] for m in MODELS]

# Load once
oof_models = [load_pred(m[1]) for m in MODELS]
test_models = [load_pred(m[2]) for m in MODELS]

# Single-model scores
single_scores = {
    name: balanced_accuracy_score(y, mat.argmax(axis=1))
    for name, mat in zip(names, oof_models)
}

for name, score in single_scores.items():
    print(f"{name} alone: {score:.6f}")

# Optional correlation heatmap
corr = np.corrcoef([mat[:, 0] for mat in oof_models])
fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(corr, annot=True, fmt=".3g", ax=ax)
ax.set_title("Heatmap of out-of-fold predictions")
plt.show()


def objective(trial):
    raw = np.array(
        [trial.suggest_float(f"w_{name}", -3.0, 3.0) for name in names],
        dtype=np.float32
    )

    weights = softmax(raw)
    blended_oof = weighted_blend(weights, oof_models)
    pred_labels = blended_oof.argmax(axis=1)

    return balanced_accuracy_score(y, pred_labels)


# Phase 1 -> QMC
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.QMCSampler(
        seed=config.SEED,
        qmc_type="sobol",
        scramble=True,
        independent_sampler=optuna.samplers.RandomSampler(seed=config.SEED),
        warn_asynchronous_seeding=True,
        warn_independent_sampling=True,
    ),
)

study.optimize(
    objective,
    n_trials=1000,
    show_progress_bar=True,
    n_jobs=1
)

# Phase 2 -> TPE
study.sampler = optuna.samplers.TPESampler(
    n_ei_candidates=64,
    seed=config.SEED,
    multivariate=False,
    warn_independent_sampling=True,
    n_startup_trials=0,
    constant_liar=True,
)

study.optimize(
    objective,
    n_trials=5000,
    show_progress_bar=True,
    n_jobs=-1
)

# Best weights
best_raw = np.array(
    [study.best_params[f"w_{name}"] for name in names],
    dtype=np.float32
)
best_w = softmax(best_raw)

# Final blend
blend_oof = weighted_blend(best_w, oof_models)
blend_test = weighted_blend(best_w, test_models)

best_single = max(single_scores.values())

print(f"\n{'=' * 50}")
print(f"Best blend: {study.best_value:.6f}")
print(f"Gain: +{study.best_value - best_single:.6f}")
print(f"\n{'=' * 50}")

for name, w in zip(names, best_w):
    print(f"{name} weight: {w:.4f} ({w:.1%})")

In [ ]:
RUN_NAME = "blend_v2_seed42"

# Save OOF probabilities
blend_oof_df = pd.DataFrame(
    blend_oof,
    columns=config.PROB_COLS
)
blend_oof_df.insert(0, "id", train_ids)

# Save Test probabilities
blend_test_df = pd.DataFrame(
    blend_test,
    columns=config.PROB_COLS
)
blend_test_df.insert(0, "id", test_ids)

oof_proba_path = Path(
    config.OOF_PROBA_DIR,
    f"{RUN_NAME}_oof_proba.csv"
)

test_proba_path = Path(
    config.TEST_PROBA_DIR,
    f"{RUN_NAME}_test_proba.csv"
)

blend_oof_df.to_csv(oof_proba_path, index=False)
blend_test_df.to_csv(test_proba_path, index=False)

# Sanity check
sanity_oof = pd.read_csv(oof_proba_path)

sanity_pred = (sanity_oof[config.PROB_COLS].to_numpy().argmax(axis=1))
print("Sanity score:", balanced_accuracy_score(y,sanity_pred))

In [ ]:
# Saving submission file as class labels
test_proba = pd.read_csv(test_proba_path)

# class with highest probability
pred_class_idx = test_proba[config.PROB_COLS].to_numpy().argmax(axis=1)

# Integer encoding -> class labels
pred_class_labels = enc.inverse_transform(pred_class_idx)

submission_df = pd.DataFrame({
    "id": test_ids,
    "PitNextLap": pred_class_labels
})

submission_df.to_csv(Path(config.SUBMISSION_DIR, f"{RUN_NAME}_submission.csv"),index=False)